In [66]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import sys
from helper.FordChatPrompts.AIFordPrompts import database_assistant_system_message, ollama_system_message

## ENV SETUP

In [67]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [68]:
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

In [69]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openai = OpenAI()

gpt_model = "gpt-4.1-mini"
ollama_model = "llama3.2"
anthropic_model = "claude-3-5-sonnet-20240620"

In [70]:
# Load Ford inventory data from JSON file
with open('Test_Data/Ford.json', 'r') as f:
    ford_inventory = json.load(f)

print("Ford inventory loaded successfully")
print(f"Total vehicles available: {len(ford_inventory)}")
print(f"Available vehicle IDs: {list(ford_inventory.keys())}")


Ford inventory loaded successfully
Total vehicles available: 20
Available vehicle IDs: ['FORD001', 'FORD002', 'FORD003', 'FORD004', 'FORD005', 'FORD006', 'FORD007', 'FORD008', 'FORD009', 'FORD010', 'FORD011', 'FORD012', 'FORD013', 'FORD014', 'FORD015', 'FORD016', 'FORD017', 'FORD018', 'FORD019', 'FORD020']


In [71]:
def filter_cars_by_criteria(model=None, max_price=None, 
                                     city=None, country=None, color=None, 
                                     year=None, edition=None):
    """Filter Ford vehicles by multiple criteria"""
    results = []
    for car in ford_inventory.values():
        if model and car["model"] != model:
            continue
        if max_price and car["price"] > max_price:
            continue
        if city and car["available_city"] != city:
            continue
        if country and car["country"] != country:
            continue
        if color and car["color"] != color:
            continue
        if year and car["year"] != year:
            continue
        if edition and car["edition"] != edition:
            continue
        results.append(car)
    return results

In [72]:
# filter_cars_by_criteria(model="Mustang")

In [73]:
filter_function = {
    "name": "filter_cars_by_criteria",
    "description": (
        "Search Ford inventory using one or more filters. "
        "At least one filter should be provided."
    ),
    "parameters": {
        "type": "object",

        "properties": {
            "model": {
                "type": "string",
                "description": "Ford model name, e.g. Mustang, Bronco, F-150"
            },

            "year": {
                "type": "integer",
                "description": "Vehicle year"
            },

            "edition": {
                "type": "string",
                "description": "Vehicle edition, e.g. Normal, GT, Raptor"
            },

            "color": {
                "type": "string",
                "description": "Vehicle color"
            },

            "country": {
                "type": "string",
                "description": "Country where the vehicle is available"
            },

            "city": {
                "type": "string",
                "description": "City where the vehicle is available"
            },

            "max_price": {
                "type": "number",
                "description": "Maximum vehicle price"
            }
        },

        "required": [],

        "additionalProperties": False
    },

    "strict": False
}

In [74]:
tools = [{"type": "function", "function": filter_function}]

In [75]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": ollama_system_message}] + history + [{"role": "user", "content": message}]
    print(messages)
    response = openai.chat.completions.create(model=gpt_model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=gpt_model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [76]:
def handle_tool_calls(message):
 
    tool_responses = []
    
    if not message.tool_calls:
        return tool_responses
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        # Execute the function with the provided arguments
        if function_name == "filter_cars_by_criteria":
            result = filter_cars_by_criteria(**function_args)
        else:
            result = {"error": f"Unknown function: {function_name}"}
        
        # Format the result as a tool response
        tool_response = {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "content": json.dumps(result)
        }
        tool_responses.append(tool_response)
    
    return tool_responses

In [78]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.
